# MatryoshkaConflictology: what it does, how, and why

This notebook explains `MatryoshkaConflictology` (ADR-024, `docs/ADRs/024_matryoshka_conflictology.md`;
class contract `docs/CICs/MatryoshkaConflictology.md`). It uses the package code itself, and it draws every
plot when it is run. No images are stored in the repository: run the notebook to see them.

**Part 1** runs on a small synthetic grid, in seconds, anywhere. **Part 2** repeats the key pictures on the
real data (the calibration input pull). It runs only if you set `DATA_PATH` below. It takes a few minutes and
about 5 GB of memory per violence type.

## The idea in one paragraph

Conflictology forecasts a grid cell by drawing at random from that cell's own last 36 months. Matryoshka
asks one extra question: **which set of past observations should a cell's forecast draw from?** Each draw
is still a value that was really observed. It is picked in three steps:

1. **Pick a doll.** Each doll is a set of cells, and they nest like Russian dolls:
   - the **cell** itself;
   - its **neighbourhood**: the cells within 5 grid cells, closer ones more likely;
   - the **rest of its country**;
   - the **rest of the map**.

   Doll *k* is picked with probability ∝ `weight[k] · exp(−h / memory[k])`. Here *h* is how far ahead the
   forecast looks, in months after the last observed month.
2. **Pick a month** within that doll, ∝ `exp(−age / memory[k])`: recent months more likely, at the doll's
   own pace.
3. **Pick a cell** within that doll: in the neighbourhood ∝ `exp(−distance / ℓ)`, and uniformly in the
   country and the map.

Each doll forgets at its own pace. Where the inner dolls forget faster, the further ahead the forecast
looks, the more it draws from the surroundings: it **opens outward**. The 8 settings ("knobs") per violence type are fitted by backtest,
not chosen by hand. Plain conflictology with a moving window is the special case where only the cell
counts, nothing fades, and the history is 36 months long.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LogNorm

from views_baseline.model.models.distributional import MatryoshkaConflictology
from views_baseline.model.models.distributional.matryoshka.doll_counts import (
    CELL, COUNTRY, DOLLS, MAP, NEAR, build_doll_counts, shell_sizes,
)
from views_baseline.model.models.distributional.matryoshka.draw import (
    build_sampler, build_shell_tables,
)
from views_baseline.model.models.distributional.matryoshka.forecast_probability import (
    doll_shares, time_weights,
)
from views_baseline.model.models.distributional.matryoshka.geometry import (
    N_COLS, OFFSET_DISTANCE, build_geometry,
)
from views_baseline.model.models.distributional.matryoshka.size_classes import (
    LABELS, N_CLASSES, classify,
)

# Part 2 only: the calibration input pull (parquet indexed month_id × priogrid_id, with lr_*_best and
# c_id). Leave as None to run Part 1 only.
DATA_PATH = os.environ.get("MATRYOSHKA_INPUT")      # or e.g. "/path/to/input.parquet"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

## Part 1: how it works, on a synthetic grid

The grid has 30 × 40 cells, three countries, a lake (sea cells are not land and not in the data), and one
small island without a country. The data has a few conflict hotspots that drift over time and spread to
nearby cells. That is exactly the pattern Matryoshka is built to pick up.

In [ ]:
ROWS, COLS = np.arange(100, 130), np.arange(300, 340)
TRAIN_END, TEST = 456, (457, 492)
months = np.arange(121, 493)                       # 1990-01 .. 2020-12, like the real data

rr, cc = np.meshgrid(ROWS, COLS, indexing="ij")
lake = (np.hypot(rr - 112, cc - 325) < 3.5)
cells = np.sort((rr * N_COLS + cc + 1)[~lake])
r, c = (cells - 1) // N_COLS, (cells - 1) % N_COLS
country = np.where(c < 313, 11, np.where(r < 116, 22, 33))
island = cells == 128 * N_COLS + 338 + 1
country[island] = 0                                  # no country: only the map is outside it

rng = np.random.default_rng(3)
centres = np.array([[105.0, 305.0], [122.0, 330.0], [108.0, 333.0]])
values = np.zeros((months.size, cells.size))
for s in range(months.size):
    centres += rng.normal(0, 0.35, centres.shape)
    centres = np.clip(centres, [100, 300], [129, 339])
    active = np.array([1.0, 1.0 if months[s] > 250 else 0.0, 1.0 if months[s] > 400 else 0.0])
    d = np.min(np.hypot(r[None] - centres[:, :1], c[None] - centres[:, 1:])
               + np.where(active[:, None] > 0, 0, 1e9), axis=0)
    rate = 4.0 * np.exp(-d / 1.5) + 0.002
    values[s] = rng.poisson(rate) * rng.integers(1, 30, cells.size) * (rng.random(cells.size) < 0.6)

geo = build_geometry(cells, country)
print(f"{cells.size} land cells, {np.unique(country[country > 0]).size} countries, "
      f"{(country <= 0).sum()} cell(s) without a country; {months.size} months; "
      f"{(values > 0).mean():.1%} of cell-months have deaths")


def to_grid(x):
    # Cell values as a 2-D image of the synthetic grid (NaN = sea).
    img = np.full((ROWS.size, COLS.size), np.nan)
    img[r - ROWS[0], c - COLS[0]] = x
    return img

### The four dolls of one cell

Every doll is a **shell**: it leaves out the dolls inside it, so no cell is counted twice. Below are the
shells of the marked cell. The neighbourhood is coloured by how likely each neighbour is to be picked,
`exp(−d / ℓ)`, with the length scale fitted for state-based violence (ℓ = 0.80 cells = 0.40°).

In [ ]:
def shells_of(i, length):
    # 0 cell, 1 neighbourhood, 2 rest of country, 3 rest of map; and the neighbour pick weights.
    which = np.full(cells.size, MAP)
    if country[i] > 0:
        which[country == country[i]] = COUNTRY
    nb = geo.neighbours[i]
    present = nb >= 0
    which[nb[present]] = NEAR
    which[i] = CELL
    pull = np.zeros(cells.size)
    pull[nb[present]] = np.exp(-OFFSET_DISTANCE[present] / length)
    return which, pull / pull.sum()

i0 = int(np.argmin(np.hypot(r - 110, c - 312)))       # a cell near a border
which, pull = shells_of(i0, 0.80)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), constrained_layout=True)
ax[0].imshow(to_grid(which), cmap=plt.matplotlib.colors.ListedColormap(
    ["black", "#d6604d", "#92c5de", "#f7f7f7"]), vmin=-0.5, vmax=3.5, origin="lower")
ax[0].set_title("the dolls of the marked cell: cell (black), neighbourhood (red),\n"
                "rest of its country (blue), rest of the map (grey); white = sea", fontsize=9)
im = ax[1].imshow(to_grid(np.where(pull > 0, pull, np.nan)), cmap="Reds", origin="lower")
ax[1].set_title("within the neighbourhood: P(pick this neighbour)", fontsize=9)
fig.colorbar(im, ax=ax[1], shrink=0.8)
for a in ax:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
sizes = shell_sizes(geo)[1][:, i0]
print("shell sizes (cells):", dict(zip(DOLLS, sizes.tolist())))

### Step 3 in the neighbourhood: a smooth distance kernel

The neighbourhood replaced two square rings of the first version (#139): the rings left square halos in the
maps. Each neighbour is picked ∝ `exp(−d / ℓ)`. The plot shows the *pull* of one neighbour relative to the
cell itself (= 1), with the fitted weights and length scales from the real data (ADR-024, amendment table).

In [ ]:
FITTED = {   # ADR-024, amendment 2026-11-06 (reports/matryoshka/PROTOTYPE_v5_kernel.md)
    "state-based": dict(weight=[1, 0.948, 0.177, 0.0456], memory=[13.5, 13.4, 11.6, 24.8], length=0.80),
    "non-state":   dict(weight=[1, 2.797, 0.380, 0.209], memory=[15.6, 11.6, 80.1, 29.1], length=1.35),
    "one-sided":   dict(weight=[1, 1.722, 0.473, 0.0279], memory=[23.1, 13.8, 22.1, 600.0], length=0.98),
}
d = np.linspace(0.5, 5, 200)
fig, ax = plt.subplots(figsize=(7, 4))
for name, k in FITTED.items():
    z = np.exp(-OFFSET_DISTANCE / k["length"]).sum()
    ax.plot(d, k["weight"][NEAR] * np.exp(-d / k["length"]) / z,
            label=f"{name}: ℓ = {k['length']:.2f} cells ({k['length'] / 2:.2f}°)")
ax.set_yscale("log"); ax.set_ylim(1e-5, 1)
ax.set_xlabel("distance to the neighbour (grid cells of 0.5°)")
ax.set_ylabel("pull of one neighbour, relative to the cell")
ax.set_title("How strongly one neighbour is drawn from, by distance (one month ahead)")
ax.legend(fontsize=8);

### Step 1: how the mix of dolls changes with distance

`P(doll | h) ∝ weight · exp(−h / memory)`, where the fitted knobs decide the shape. The plot shows each
violence type's fitted knobs, for a cell with all four dolls present.

- **State-based:** the cell, the neighbourhood and the country forget at about the same pace (12–14
  months), and the map more slowly (25). The forecast opens outward, but only gently.
- **Non-state:** the neighbourhood forgets fast (12 months) and the country slowly (80). Three years ahead
  the forecast is drawn mostly from the rest of the country.
- **One-sided:** the neighbourhood forgets faster (14 months) than the cell (23). So the
  cell's own share *grows* with distance, and the world backstop (600 months) slowly gains too.

"Opens outward" is therefore what the data says for state-based and non-state violence. For one-sided
violence, the neighbourhood's say moves both inward to the cell and outward.

In [ ]:
h = np.arange(1, 61)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True, constrained_layout=True)
for ax, (name, k) in zip(axes, FITTED.items()):
    a = np.array(k["weight"])[:, None] * np.exp(-h[None] / np.array(k["memory"])[:, None])
    ax.stackplot(h, a / a.sum(0), labels=DOLLS, colors=["black", "#d6604d", "#92c5de", "#bbbbbb"])
    ax.set_title(name); ax.set_xlabel("months ahead (h)")
axes[0].set_ylabel("P(doll | h)"); axes[0].legend(loc="lower left", fontsize=8);

### Step 2: which months a doll draws from

Within a doll, month *s* is picked ∝ `exp(−(c − s) / memory)`, divided by its total (`Z`) so that each
doll's total say stays its weight, whatever its memory. The one-sided map memory sits at the 600-month
ceiling: "the whole history, roughly equally" (#140, accepted).

In [ ]:
age = np.arange(0, 336)
fig, ax = plt.subplots(figsize=(7, 4))
for doll, m in zip(DOLLS, FITTED["state-based"]["memory"]):
    w = np.exp(-age / m); ax.plot(age, w / w.sum(), label=f"{doll} (memory {m} months)")
w = np.exp(-age / 600); ax.plot(age, w / w.sum(), "--", label="one-sided map (600, the ceiling)")
ax.set_yscale("log"); ax.set_xlabel("age of the month (months before the last observed one)")
ax.set_ylabel("P(pick this month)"); ax.set_title("Step 2, state-based knobs"); ax.legend(fontsize=8);

### The fit: backtesting on the training data

The knobs are found by **backtest**. The model pretends the history ended at an earlier month (four
cutoffs, a year apart), forecasts the 36 months after each, and scores the forecasts. The score is the
mean log probability of the observed **size class** (0 · 1 · 2–5 · 6–25 · 26–100 · 101–1,000 · >1,000
deaths). It is computed exactly, without sampling, together with its exact slope, and L-BFGS-B searches
the 8 knobs. On the synthetic data this takes seconds.

In [ ]:
index = pd.MultiIndex.from_product([months, cells], names=["month_id", "priogrid_id"])
df = pd.DataFrame({"y": values.ravel(), "c_id": np.tile(country, months.size).astype(float)},
                  index=index)
model = MatryoshkaConflictology(
    targets=["y"], partition_dict={"test": TEST}, loa="pgm", n_samples=256, country_column="c_id",
    backtest_cutoffs=4, backtest_horizon=36, stability_refits=False, seed=42).fit(df)
k = model.knobs["y"]
print(f"converged: {k['converged']} after {k['steps']} steps ({k['seconds']} s)")
print(f"score {k['score']:.4f} (starting values {k['start_score']:.4f})")
print(pd.DataFrame({"weight": k["weight"], "memory (months)": np.round(k["memory"], 1)},
                   index=DOLLS))
print(f"neighbourhood length scale ℓ = {k['length_scale']:.2f} cells")

The synthetic hotspots ignore borders, so the fit switches the country doll off (weight ≈ 1e-6) and leans on
the neighbourhood. On the real data the country does carry weight (Part 2). The knobs are a finding about
the data, not settings.

### What predict draws is exactly what the fit scores

For one cell and one forecast distance, 200,000 draws from the package's sampler are compared with the
closed-form class probabilities the fit uses. The tests check this too
(`test_sampler_matches_the_exact_probabilities`).

In [ ]:
c_hist = TRAIN_END
hist = months <= c_hist
classes = classify(values)
dc = build_doll_counts(classes[hist], months[hist], geo)
tables = build_shell_tables(geo)
sampler = build_sampler(values[hist].astype(np.float32), months[hist], c_hist, k, geo, tables)

def class_probabilities(knobs, h_values):
    # Exact P(class | cell, h) for every cell: (H, N, K).
    q = np.stack([doll_shares(dc, time_weights(months[hist], np.array([c_hist]), knobs["memory"][j]),
                              j, knobs["length_scale"])[0] for j in range(len(DOLLS))])
    p_doll = np.diff(np.concatenate([np.zeros((cells.size, len(h_values), 1)),
                                     sampler.doll_cdf(np.arange(cells.size), np.asarray(h_values))],
                                    axis=2), axis=2)                          # (N, H, 4)
    return np.einsum("nhd,dnk->hnk", p_doll, q)

i1, S = int(np.argmax(values[hist][-24:].sum(0))), 200_000
draws = sampler.draw(np.array([i1]), np.array([12]), np.random.default_rng(1).random((1, 1, 3, S)))
freq = np.bincount(classify(draws.ravel()), minlength=N_CLASSES) / S
exact = class_probabilities(k, [12])[0, i1]
fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(N_CLASSES)
ax.bar(x - 0.2, exact, 0.4, label="exact (what the fit scores)")
ax.bar(x + 0.2, freq, 0.4, label="200,000 draws (what predict returns)")
ax.set_xticks(x, LABELS); ax.set_yscale("log"); ax.set_xlabel("deaths in the month (size class)")
ax.set_title(f"One cell, 12 months ahead: largest gap {np.abs(freq - exact).max():.4f}")
ax.legend(fontsize=8);

### Matryoshka against conflictology, out of sample

Below are the chance of any deaths one month and three years after the training data ends, next to plain
conflictology, whose chance is the share of violent months in the cell's last 36. Crosses mark the cells
that actually had deaths in the 36 forecast months.

- **Conflictology** gives exactly 0 to every cell whose window was quiet, whatever happens around it, and
  its map is the same one month and three years ahead.
- **Matryoshka** spreads chance to the surroundings, and more so the further ahead it looks.

In [ ]:
P = class_probabilities(k, np.arange(1, 37))                  # (36, N, K)
p_any = 1 - P[..., 0]
conflictology = (values[(months > c_hist - 36) & hist] > 0).mean(0)
future = values[(months > c_hist) & (months <= c_hist + 36)]
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4), constrained_layout=True)
norm = LogNorm(1e-4, 1)
for ax, img, title in zip(axes, [conflictology, p_any[0], p_any[35]],
                          ["conflictology (any month ahead)", "Matryoshka, 1 month ahead",
                           "Matryoshka, 36 months ahead"]):
    im = ax.imshow(to_grid(np.maximum(img, 1e-4)), norm=norm, cmap="magma_r", origin="lower")
    hit = (future > 0).any(0)
    ax.plot(c[hit] - COLS[0], r[hit] - ROWS[0], "x", color="#1a9850", ms=3, alpha=0.6)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.grid(False)
fig.colorbar(im, ax=axes, shrink=0.8, label="P(any deaths in the month)")

obs = classify(future)                                        # (36, N)
p_obs = np.take_along_axis(P, obs[..., None].astype(int), axis=2)[..., 0]
conf_p_any = np.broadcast_to(conflictology, obs.shape)
conf_zero = ((obs > 0) & (conf_p_any == 0)).sum()
print(f"cell-months with deaths that conflictology gave exactly 0%: {conf_zero} "
      f"of {(obs > 0).sum()}; Matryoshka's smallest chance for an observed class: {p_obs.min():.1e}")

### Plain conflictology is a special case

With weight only on the cell, infinite memories (nothing fades) and a history of exactly 36 months, every
draw is uniform over the cell's last 36 values. That is `RollingConflictology`. Matryoshka adds exactly two
ideas on top of it: the outer dolls and the fading.

In [ ]:
special = MatryoshkaConflictology(
    targets=["y"], partition_dict={"test": TEST}, loa="pgm", n_samples=4000, country_column="c_id",
    backtest_cutoffs=4, backtest_horizon=36, stability_refits=False, seed=0)
special.knobs = {"y": {"weight": [1, 0, 0, 0], "memory": [np.inf] * 4, "length_scale": 1.0}}
special.entity_ids, special.country, special.history_start = cells, country, TRAIN_END - 35
window = df[(df.index.get_level_values("month_id") > TRAIN_END - 36)
            & (df.index.get_level_values("month_id") <= TRAIN_END)]
out = special.predict(window, 0, 1)["y"].values
gap = max(abs((out[j] > 0).mean() - (values[(months > TRAIN_END - 36) & hist, j] > 0).mean())
          for j in range(cells.size))
p = (values[(months > TRAIN_END - 36) & hist] > 0).mean(0)
z = [abs((out[j] > 0).mean() - p[j]) / np.sqrt(p[j] * (1 - p[j]) / out.shape[1])
     for j in range(cells.size) if 0 < p[j] < 1]
print(f"largest gap between the special case's P(any) and conflictology's, over all cells: {gap:.3f}; "
      f"in standard errors: {max(z):.1f} (the largest of {len(z)} noisy estimates is expected around "
      f"{np.sqrt(2 * np.log(len(z))):.1f}). Cells conflictology gives exactly 0 or 1 match exactly: "
      f"{all((out[j] > 0).mean() == p[j] for j in range(cells.size) if p[j] in (0, 1))}")

## Part 2: the real data

Set `DATA_PATH` in the setup cell, or the environment variable `MATRYOSHKA_INPUT`, to the calibration
input pull (64,818 land cells, months 121–504, with `lr_sb_best`, `lr_ns_best`, `lr_os_best` and `c_id`).
The fit uses months ≤ 456 only. The maps show the out-of-sample forecast from December 2017 (month 456)
for months 457 (1 month ahead) and 492 (3 years ahead). The fitted knobs equal those in ADR-024's
amendment table (`reports/matryoshka/package_check.py` checks this).

Each violence type takes about 3.5 minutes and about 4–5 GB to fit. `REAL_TARGETS` picks which ones.

In [ ]:
REAL_TARGETS = ["lr_sb_best"]          # add "lr_ns_best", "lr_os_best" for all three
if DATA_PATH is None:
    print("DATA_PATH is not set: skipping Part 2.")
else:
    real = pd.read_parquet(DATA_PATH, columns=REAL_TARGETS + ["c_id"])
    real = real[real.index.get_level_values("month_id") <= 492]
    rmodel = MatryoshkaConflictology(
        targets=REAL_TARGETS, partition_dict={"test": (457, 492)}, loa="pgm", n_samples=256,
        country_column="c_id", backtest_cutoffs=4, backtest_horizon=36, stability_refits=False,
        seed=42).fit(real)
    for t in REAL_TARGETS:
        kk = rmodel.knobs[t]
        print(f"\n{t}: score {kk['score']:.5f}, converged {kk['converged']}, ℓ = "
              f"{kk['length_scale']:.2f} cells ({kk['length_scale'] / 2:.2f}°)")
        print(pd.DataFrame({"weight": np.round(kk["weight"], 4),
                            "memory (months)": np.round(kk["memory"], 1)}, index=DOLLS))

In [ ]:
def real_class_probabilities(t, h_values):
    # Exact P(class | cell, h) at c = 456 for every land cell: (H, N, K), plus what is needed below.
    months_r = np.sort(real.index.get_level_values("month_id").unique())
    cells_r = np.sort(real.index.get_level_values("priogrid_id").unique())
    wide = real[t].unstack("priogrid_id").reindex(index=months_r, columns=cells_r).to_numpy()
    hist_r = months_r <= 456
    geo_r = build_geometry(cells_r, rmodel.country)
    dc_r = build_doll_counts(classify(wide[hist_r]), months_r[hist_r], geo_r)
    kk = rmodel.knobs[t]
    q = np.stack([doll_shares(dc_r, time_weights(months_r[hist_r], np.array([456]), kk["memory"][j]),
                              j, kk["length_scale"])[0] for j in range(len(DOLLS))])
    hv = np.asarray(h_values, dtype=float)
    a = np.array(kk["weight"])[:, None] * np.exp(-hv[None] / np.array(kk["memory"])[:, None])
    a = a[:, :, None] * (dc_r.sizes > 0)[:, None, :]                      # (4, H, N)
    P_r = np.einsum("dhn,dnk->hnk", a / a.sum(0), q)
    return P_r, wide, months_r, cells_r


def world_image(cells_r, x):
    img = np.full((360, 720), np.nan)
    img[(cells_r - 1) // 720, (cells_r - 1) % 720] = x
    return img


def show_map(ax, cells_r, x, norm, title, crop=None, cmap="magma_r"):
    im = ax.imshow(world_image(cells_r, x), norm=norm, cmap=cmap, origin="lower",
                   extent=(-180, 180, -90, 90), interpolation="nearest")
    if crop:
        ax.set_xlim(crop[0], crop[1]); ax.set_ylim(crop[2], crop[3])
    ax.set_title(title, fontsize=9); ax.grid(False)
    return im


if DATA_PATH is not None:
    for t in REAL_TARGETS:
        P_r, wide, months_r, cells_r = real_class_probabilities(t, np.arange(1, 37))
        hist_r = months_r <= 456
        conf = (wide[hist_r][-36:] > 0).mean(0)
        future_r = classify(wide[(months_r > 456) & (months_r <= 492)])
        p_obs = np.take_along_axis(P_r, future_r[..., None].astype(int), axis=2)[..., 0]
        print(f"{t}, months 457–492: cell-months with deaths given exactly 0% by conflictology: "
              f"{int(((future_r > 0) & (conf[None] == 0)).sum())}; observed classes Matryoshka gave "
              f"< 1e-12: {int((p_obs < 1e-12).sum())}")
        norm = LogNorm(1e-6, 1)
        fig, axes = plt.subplots(3, 1, figsize=(13, 17), constrained_layout=True)
        for ax, x, title in zip(axes, [conf, 1 - P_r[0, :, 0], 1 - P_r[35, :, 0]],
                                ["conflictology: share of violent months in 425–456",
                                 "Matryoshka: P(any deaths), month 457 (1 month ahead)",
                                 "Matryoshka: P(any deaths), month 492 (36 months ahead)"]):
            im = show_map(ax, cells_r, np.maximum(x, 1e-6), norm, f"{t}: {title}")
        fig.colorbar(im, ax=axes, shrink=0.5, label="P(any deaths in the month)")
        plt.show()

### How big, if anything happens?

The chance that a month with deaths has more than 25 deaths: `P(class ≥ 26–100) / P(any deaths)`. This is
exact, from the same class probabilities.

In remote quiet places (Canada, Siberia, Australia) nearly every violent draw comes from the rest of the
map. The map's violence is dominated by ongoing wars, so the size of a possible outbreak there looks
war-sized. This is the honest answer of the world pool. Separating "whether" from "how big" was considered
and rejected (#137; ADR-024 amendment).

In [ ]:
if DATA_PATH is not None:
    for t in REAL_TARGETS:
        P_r, wide, months_r, cells_r = real_class_probabilities(t, [1])
        big = P_r[0, :, 4:].sum(1) / np.maximum(1 - P_r[0, :, 0], 1e-300)
        fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
        im = show_map(axes[0], cells_r, big, plt.Normalize(0, 0.5), f"{t}: world", cmap="viridis")
        show_map(axes[1], cells_r, big, plt.Normalize(0, 0.5), f"{t}: Africa and the Middle East",
                 crop=(-20, 60, -36, 40), cmap="viridis")
        fig.colorbar(im, ax=axes, shrink=0.8,
                     label="P(more than 25 deaths | any deaths), month 457")
        plt.show()

## Where to read more

- **ADR-024**, `docs/ADRs/024_matryoshka_conflictology.md`: the design, and the amendment of 2026-11-06
  with every decision from the prototype (kernel kept; province, size split, neighbouring countries,
  latitude and smoothing dropped).
- **The class contract**, `docs/CICs/MatryoshkaConflictology.md`: the guarantees, failure modes and tests.
- **The running log** of the prototype, `reports/matryoshka/LOG.md`, with the numbers behind each decision.